In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece torch accelerate triton peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.8 MB/s eta 0:00:00


In [ ]:
from typing import List, Dict
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu import corpus_bleu, corpus_chrf
from tqdm.auto import tqdm
from peft import PeftModelForSeq2SeqLM

In [ ]:
TEST_PATH = "/content/test.csv"  # change if needed

test_df = pd.read_csv(TEST_PATH)

assert {"text", "gloss"}.issubset(test_df.columns), \
    "Test CSV must contain src_text and tgt_gloss columns"

print("Test samples:", len(test_df))
test_df.head()


Test samples: 159


,text,gloss
0,আমি বিস্কুট খাই,আমি বিস্কুট খাওয়া
1,তুমি কোথায় যাও?,তুমি কোথায় যাও
2,তুমি কেমন আছো?,তুমি কেমন আছ
3,দরজা জানালা বন্ধ কর,দরজা জানালা বন্ধ করা
4,আনারসটি বড়,আনারসটি বড়


In [ ]:
def generate_predictions(
    model: AutoModelForSeq2SeqLM,
    tokenizer: AutoTokenizer,
    inputs: List[str],
) -> List[str]:
    model.eval()
    outputs: List[str] = []

    for i in tqdm(range(0, len(inputs), BATCH_SIZE)):
        batch = inputs[i:i + BATCH_SIZE]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(DEVICE)

        with torch.no_grad():
            generated_ids = model.generate(
                **encoded,
                max_length=MAX_GEN_LEN,
                num_beams=4,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )
        outputs.extend(decoded)

    return outputs


In [ ]:
def evaluate_predictions(
    predictions: List[str],
    references: List[str],
) -> Dict[str, float]:
    bleu = corpus_bleu(predictions, [references])
    chrf = corpus_chrf(predictions, [references])

    return {
        "BLEU": bleu.score,
        "chrF": chrf.score,
    }


In [ ]:
!unzip /content/drive/MyDrive/Sign_language_T2G_Paper/model_weights/marian-lora-fold-1 -d /content/fold1
!unzip /content/drive/MyDrive/Sign_language_T2G_Paper/model_weights/marian-lora-fold-2 -d /content/fold2
!unzip /content/drive/MyDrive/Sign_language_T2G_Paper/model_weights/marian-lora-fold-3 -d /content/fold3
!unzip /content/drive/MyDrive/Sign_language_T2G_Paper/model_weights/marian-lora-fold-4 -d /content/fold4
!unzip /content/drive/MyDrive/Sign_language_T2G_Paper/model_weights/marian-lora-fold-5 -d /content/fold5

In [ ]:
BATCH_SIZE = 8
MAX_GEN_LEN = 128

In [ ]:
MODEL_DIRS = [
    "/content/fold1",
    "/content/fold2",
    "/content/fold3",
    "/content/fold4",
    "/content/fold5",
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
results = []

for fold_idx, base_model_dir in enumerate(MODEL_DIRS, start=1):
    print(f"\n===== Evaluating Fold {fold_idx} =====")

    # Construct the correct path to the model and tokenizer files
    model_dir = f"{base_model_dir}/content/marian-lora-fold-{fold_idx}"

    # model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(DEVICE)
    base_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-inc").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = PeftModelForSeq2SeqLM.from_pretrained(base_model, model_dir)

    preds = generate_predictions(
        model=model,
        tokenizer=tokenizer,
        inputs=test_df["text"].tolist()
    )

    scores = evaluate_predictions(
        predictions=preds,
        references=test_df["gloss"].tolist()
    )

    scores["fold"] = fold_idx
    results.append(scores)

    # free VRAM
    del model
    torch.cuda.empty_cache()

In [ ]:
results_df = pd.DataFrame(results)
results_df


,BLEU,chrF,fold
0,4.796972,45.574578,1
1,6.122319,45.174096,2
2,7.115599,44.595214,3
3,5.862983,45.202522,4
4,7.402131,45.812714,5


In [ ]:
print("Mean scores across folds:")
results_df.mean(numeric_only=True)


Mean scores across folds:


,0
BLEU,6.260001
chrF,45.271825
fold,3.000000


In [ ]:
import os
OUTPUT_DIR = "/content/eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for fold_idx, base_model_dir in enumerate(MODEL_DIRS, start=1):
    # Construct the correct path to the model and tokenizer files
    model_dir = f"{base_model_dir}/content/marian-lora-fold-{fold_idx}"

    base_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-inc").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = PeftModelForSeq2SeqLM.from_pretrained(base_model, model_dir)

    preds = generate_predictions(
        model,
        tokenizer,
        test_df["text"].tolist()
    )

    out_df = test_df.copy()
    out_df["prediction"] = preds
    out_df.to_csv(f"{OUTPUT_DIR}/fold_{fold_idx}_predictions.csv", index=False)

    del model
    torch.cuda.empty_cache()

pytorch_model.bin:   0%|          | 0.00/305M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/305M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (9 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


  0%|          | 0/20 [00:00<?, ?it/s]